# Feature Selection Example using MotherSettings Class

Here, a simple example is shown how the default feature selection pipeline works. Furthermore, the feature selector can be analyzed to see which features have been removed by which selector.

In [1]:
from mother.settings import MotherSettings
import pandas as pd
from sklearn.compose import ColumnTransformer
from mother import pipeline_utils as mother_takes_care
from sklearn.datasets import make_classification

import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
settings: MotherSettings = MotherSettings.create()
settings.model.feature_selection_flags = [
    "DROP_CONSTANT",
    "DROP_CORRELATED",
    "DROP_DUPLICATES",
    "DROP_UNIMPORTANT",
]
settings.pipeline.remainder = "passthrough"
settings.model.feature_selection_type = "catboost"  # permutation does not work yet
settings.model.categorical_features = ["categorical_A"]
data: pd.DataFrame = pd.DataFrame(
    {
        "constant_A": [1, 1],
        "constant_B": [3, 3],
        "correlated_A": [1, 2],
        "correlated_B": [1, 2],
        "remaining_feature": [10, 9],
        "dupl_A": [1, 0],
        "dupl_B": [1, 0],
        "categorical_A": ["cat", None],
    }
)
X, y = make_classification(
    n_samples=1000,
    n_features=12,
    n_redundant=4,
    n_clusters_per_class=1,
    weights=[0.50],
    class_sep=2,
    random_state=1,
)
# transform arrays into pandas df and series
colnames = ["var_" + str(i) for i in range(12)]
data = pd.DataFrame(X, columns=colnames)
categorical_ix = data.select_dtypes(include=["object", "bool"]).columns.tolist()
settings.model.categorical_features = categorical_ix

data.head()

,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,var_10,var_11
0,1.471061,-2.376400,-0.247208,1.210290,-3.247521,0.091527,3.686599,-2.230170,2.070483,3.528094,2.070526,-1.989335
1,1.819196,1.969326,-0.126894,0.034598,-2.910112,-0.186802,3.318577,-1.447490,2.421477,3.304213,1.184820,-1.309524
2,1.625024,1.499174,0.334123,-2.233844,-3.399345,-0.313881,3.861502,-2.240741,2.263546,3.717297,-0.066448,-0.852703
3,1.939212,0.075341,1.627132,0.943132,-4.783124,-0.468041,5.423008,-3.534861,2.792500,5.131589,0.713558,0.484649
4,1.579307,0.372213,0.338141,0.951526,-3.199285,0.729005,3.635738,-2.053965,2.186741,3.512742,0.398790,-0.186530


In [2]:
from sklearn.model_selection import KFold


X_train = data.iloc[:, :-1]  # ignore last column
y_train = data.iloc[:, -1:]  # select last column
cv = KFold(n_splits=5, shuffle=True)
transformer: ColumnTransformer = mother_takes_care.get_feature_selection_pipeline(
    settings=settings, data=X_train, cv=cv
)
assert transformer is not None
transformer.set_output(transform=settings.pipeline.transform)
result: pd.DataFrame = transformer.fit_transform(X_train, y_train)
result.head()

2025-06-05 06:18:30,789 - mother.pipeline_utils - INFO - Creating feature selection pipeline for numeric columns.
2025-06-05 06:18:30,791 - mother.pipeline_utils - INFO - Determine categorical and numerical features. Categorical features are skipped
2025-06-05 06:18:30,815 - mother.pipeline_utils - INFO - The default correlation method is used
2025-06-05 06:18:30,818 - mother.pipeline_utils - INFO - Boruta usage: False
2025-06-05 06:18:30,819 - mother.ml.utils - INFO - Using 'RMSE' as loss function.
2025-06-05 06:18:30,820 - mother.pipeline_utils - INFO - Setting up catboost importance feature selection
2025-06-05 06:18:33,624 - mother.ml.estimators - INFO - Finished catboost feature importance calculation
2025-06-05 06:18:33,626 - mother.ml.estimators - INFO - Feature importances have been turned into percentiles
2025-06-05 06:18:33,627 - mother.ml.estimators - INFO - Initialized catboost feature importance estimator


,var_1,var_2,var_3,var_5,var_7,var_8,var_10
0,-2.376400,-0.247208,1.210290,0.091527,-2.230170,2.070483,2.070526
1,1.969326,-0.126894,0.034598,-0.186802,-1.447490,2.421477,1.184820
2,1.499174,0.334123,-2.233844,-0.313881,-2.240741,2.263546,-0.066448
3,0.075341,1.627132,0.943132,-0.468041,-3.534861,2.792500,0.713558
4,0.372213,0.338141,0.951526,0.729005,-2.053965,2.186741,0.398790


In [3]:
transformer

ColumnTransformerWithHyperparameterRooting(remainder='passthrough',
                                           transformers=[('feature_selector',
                                                          PipelineWithHyperparameterRooting(steps=[('duplicate_selector',
                                                                                                    DropDuplicateFeatures()),
                                                                                                   ('constant_selector',
                                                                                                    DropConstantFeatures(missing_values='ignore')),
                                                                                                   ('correlation_selector',
                                                                                                    SmartCorrelatedSelection(selection_method='variance',
                                                                                                                             threshold=0.9)),
                                                                                                   ('importance_selector',
                                                                                                    MotherSelectFromModel(estimator=MotherCatboostImportance(estimator=<catboost.core.CatBoostRegressor object at 0x736941b26990>),
                                                                                                                          threshold=0.0))]),
                                                          <function get_numeric_columns at 0x73693fca0ea0>)],
                                           verbose_feature_names_out=False)

In [4]:
from mother import pipeline_utils

pipeline_utils.report_feature_selection(transformer, X_train)

2025-06-05 06:18:33,703 - mother.pipeline_utils - INFO - Analyzing feature selection pipeline
2025-06-05 06:18:33,706 - mother.pipeline_utils - INFO - Analyzing feature_selector transformer
2025-06-05 06:18:33,707 - mother.pipeline_utils - INFO - Nested pipeline found
2025-06-05 06:18:33,713 - mother.pipeline_utils - INFO - Pipeline step: duplicate_selector
2025-06-05 06:18:33,716 - mother.pipeline_utils - INFO - No features to drop
2025-06-05 06:18:33,719 - mother.pipeline_utils - INFO - Pipeline step: constant_selector
2025-06-05 06:18:33,721 - mother.pipeline_utils - INFO - No features to drop
2025-06-05 06:18:33,723 - mother.pipeline_utils - INFO - Pipeline step: correlation_selector
2025-06-05 06:18:33,727 - mother.pipeline_utils - INFO - Correlated features: {'var_7': {'var_6', 'var_4', 'var_9'}, 'var_8': {'var_0'}}
2025-06-05 06:18:33,737 - mother.pipeline_utils - INFO - Standard deviation for features:
var_7    2.159634
var_6    2.032947
var_4    1.810273
var_9    1.764249
dtyp